# TOS<sup>2</sup>CA Phenomenon Definition (PhDef) End-To-End Example

This notebook is meant to be an example of how to run a PheDef job in an end-to-end fashion.  It reads/subsets data, runs ForTraCC on the data, stitches the ForTraCC file together from chunks, and then creates mask plots.  It does not include any steps in the Data Curation stage of TOS<sup>2</sup>CA.

## Import Python Libraries

First, we'll import the necessary Python libraries.  This assumes you already have [tos2ca-anomaly-detection](https://github.com/nasa-jpl/tos2ca-anomaly-detection) and [tos2ca-fortracc-module](https://github.com/nasa-jpl/tos2ca-fortracc-module) installed and in your `PYTHONPATH`.  You should also have installed the [data dictionaries](https://github.com/nasa-jpl/tos2ca-data-dictionaries) on your system and updated any paths in the code to point to them.

In [ ]:
import sys
from tos2ca.iolib.merra2 import merra2_reader
from tos2ca.utils.plot import mask_plot
from tos2ca.utils.fortracc import callFortraccSparse, stitchFortracc

## Job Parameters

The job we're going to run has the following parameters, which you'll need to insert into your MySQL database instance's `jobs` table (see the [database architecture](../../db/tosca_db.sql) to get going on that if you haven't installed it already).

```mysql
jobID: 390
userID: 1
nChunks: 1
stage: phdef
phdefJobID: NULL
dataset: M2I1NXINT_5.12.4
variable: TQI
ST_ASTEXT(coords): POLYGON((-103.32 -1.05,-103.32 31.65,-39.34 31.65,-39.34 -1.05,-103.32 -1.05)) 
startDate: 2020-01-04 00:00:00
endDate: 2020-02-03 23:59:59
ineqOperator: anomalyEvent
ineqValue: 1
description: Example MERRA-2 PhDef Job
status: pending
submitTime: 2024-07-23 19:21:23
```

The job (390) only has one chunk for simplicty here, but jobs can have <i>n</i> number of chunks.  This also assumes you've chunked your job into chunks and have an entry for chunkID 1 in your `chunks` table.  It also requires that you have a user in your `users` table with a userID of 1.

Note that this job happens to use MERRA-2 data, which is why we imported `merra2_reader` in the imports block.

## Setup Global Variables

Again, we're using jobID #390 and chunkID #1

In [4]:
jobID = 390
chunkID = 1

## Read and Subset the Data

This will output results into Elasticache for each chunk.

In [ ]:
print("Running jobID: %s-%s" % (jobID, chunkID))
merra2_reader(jobID, chunkID)
print("Job complete")

## Run ForTraCC

This will output results into Elasticache for each chunk.

In [ ]:
print("Run ForTraCC")
callFortraccSparse(jobID, chunkID)
print("Done running ForTraCC")

## Stitch Results 

Note that there's not much to stich because there is only 1 chunk here.  If there are more than one chunks in your job that you need to stitch together, each chunk needs to have one overlapping chunk with the previous chunk.  For example, if chunkID 22 has an end time of '2025-01-01 01:00:00', then the start time of chunkID 23 should have a start time of '2025-01-01 01:00:00'.  

This stage will output:
- a single netCDF-4 file, 
- a JSON file with the hirerachy of the netCDF-4 file
- a JSON table of contents file for all phenomenon in this job

In [ ]:
print("Combine chunks into a single file")
stitchFortracc(jobID)
print("Done with ForTraCC")

## Plot the Data

Now we'll plot the data using the data from the ForTraCC chunks that should still be in Elasticache.  For each timestep, this will output:
- a PNG plot file
- a GeoJSON file with the outlines of each anomaly
These can be viwed on the TOS<sup>2</sup>CA website / [user interface](https://github.com/nasa-jpl/tos2ca-user-interface) if you have that installed.


In [ ]:
print("Plot the data")
mask_plot(jobID, chunkID)
print("Done plotting")

## Outputs

All outputs (from the stitcher and plotter) will output to the S3 bucket name that you have identified in AWS secrets (see what [secrets you need to setup](../../templates/aws_secrets.json)):

```
s3://<bucket name>/<jobID>/
```